# S&P 500 — Task-Level Wage Exposure Analysis

Estimates the total wage value of work performed in tasks classified as **E0**, **E1**, and **E2/E3**
and computes the potentially automated wage value under two parallel views:

- **Theoretical (Gemini rubric).** Applies flat automation scenarios of **40 %, 50 %, 60 %** to the
  E1 + E2/E3 wage mass. The **50 %** scenario is used as the headline summary.
- **Observed (Anthropic penetration as efficiency proxy).** For each occupation × task row, uses the
  task-level penetration value directly as an efficiency coefficient, so the dollar savings are
  `task_value × penetration`, summed over the full SP500 weighted task set.

**Pipeline**
1. Load company workforce data (`sp500_company_data.parquet`)
2. Load task-time distribution π (`job_task_time_distribution_30_0.csv`)
3. Load task-level exposure labels (`full_labelset_new.tsv`)
4. Load task penetration (`Anthropic/task_penetration.csv`)
5. Aggregate workforce → occupation-level worker counts and wages
6. Build occupation × task table and attach π, exposure labels, and penetration
7. Compute task economic value = workers × wage × π
8. Run diagnostics
9. Aggregate by exposure bucket and apply Gemini automation scenarios (40 / 50 / 60 %, headline 50 %)
10. Compute observed efficiency savings = Σ(task_value × penetration)
11. Combined summary (Gemini headline vs. Observed)

In [17]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print("Libraries loaded ✓")

Libraries loaded ✓


## 1 · Configuration — File Paths

In [18]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = "/Users/nicobagnoli/Documents/PYTHON/SMP500-1"

PARQUET_PATH   = os.path.join(BASE, "data", "sp500_company_data.parquet")
TASK_TIME_PATH = os.path.join(BASE, "data", "job_task_time_distribution_30_0.csv")
EXPOSURE_PATH  = os.path.join(BASE, "Eloundou_New", "full_labelset_new.tsv")
PENETRATION_PATH = os.path.join(BASE, "Anthropic", "task_penetration.csv")
OUTPUT_PATH    = os.path.join(BASE, "Eloundou_New", "task_value_exposure_summary.csv")

for label, path in [("Company data", PARQUET_PATH),
                    ("Task-time dist.", TASK_TIME_PATH),
                    ("Exposure labels", EXPOSURE_PATH),
                    ("Task penetration", PENETRATION_PATH)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {label:20s}  {path}")
    if not exists:
        raise FileNotFoundError(f"Required file not found: {path}")

  ✓  Company data          /Users/nicobagnoli/Documents/PYTHON/SMP500-1/data/sp500_company_data.parquet
  ✓  Task-time dist.       /Users/nicobagnoli/Documents/PYTHON/SMP500-1/data/job_task_time_distribution_30_0.csv
  ✓  Exposure labels       /Users/nicobagnoli/Documents/PYTHON/SMP500-1/Eloundou_New/full_labelset_new.tsv
  ✓  Task penetration      /Users/nicobagnoli/Documents/PYTHON/SMP500-1/Anthropic/task_penetration.csv


## 2 · Load Data

In [19]:
# ── 2.1 Company workforce data ────────────────────────────────────────────────
print("Loading company workforce data …", flush=True)
df_raw = pd.read_parquet(PARQUET_PATH)

print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {df_raw.columns.tolist()}")

# ── Column auto-detection ─────────────────────────────────────────────────────
REQUIRED = {
    "occ_id"  : ["onet_code", "O*NET-SOC Code", "onet", "soc_code"],
    "wage"    : ["salary", "wage", "annual_wage", "mean_wage"],
    "weight"  : ["weight", "weights", "n_workers", "emp_count"],
}

resolved = {}
for field, candidates in REQUIRED.items():
    found = next((c for c in candidates if c in df_raw.columns), None)
    if found is None:
        raise KeyError(
            f"Cannot find column for '{field}'. "
            f"Expected one of {candidates}. "
            f"Available columns: {df_raw.columns.tolist()}"
        )
    resolved[field] = found
    print(f"  Detected '{field}' → '{found}'")

OCC_COL    = resolved["occ_id"]
WAGE_COL   = resolved["wage"]
WEIGHT_COL = resolved["weight"]

print("\nCompany data loaded ✓")

Loading company workforce data …
  Rows: 23,244,615
  Columns: ['user_id', 'position_id', 'rcid', 'seniority', 'country', 'salary', 'onet_code', 'startdate', 'enddate', 'weight', 'highest_degree', 'sex_predicted', 'ethnicity_predicted', 'ticker', 'naics_code', 'company']
  Detected 'occ_id' → 'onet_code'
  Detected 'wage' → 'salary'
  Detected 'weight' → 'weight'

Company data loaded ✓


In [20]:
# ── 2.2 Task-time distribution ────────────────────────────────────────────────
print("Loading task-time distribution …", flush=True)
df_pi = pd.read_csv(TASK_TIME_PATH)

# Auto-detect columns
PI_OCC_CANDIDATES  = ["O*NET-SOC Code", "onet_code", "onet", "soc_code"]
PI_TASK_CANDIDATES = ["Task ID", "task_id", "TaskID"]
PI_PI_CANDIDATES   = ["pi", "Pi", "task_share", "time_share"]

pi_occ_col  = next((c for c in PI_OCC_CANDIDATES  if c in df_pi.columns), None)
pi_task_col = next((c for c in PI_TASK_CANDIDATES if c in df_pi.columns), None)
pi_pi_col   = next((c for c in PI_PI_CANDIDATES   if c in df_pi.columns), None)

for label, val, cands in [
    ("occ",  pi_occ_col,  PI_OCC_CANDIDATES),
    ("task", pi_task_col, PI_TASK_CANDIDATES),
    ("pi",   pi_pi_col,   PI_PI_CANDIDATES),
]:
    if val is None:
        raise KeyError(f"Task-time file: cannot find column for '{label}'. "
                       f"Expected one of {cands}. Got: {df_pi.columns.tolist()}")
    print(f"  Detected '{label}' → '{val}'")

# Rename to standard names for the rest of the notebook
df_pi = df_pi.rename(columns={pi_occ_col: "occ_id", pi_task_col: "task_id", pi_pi_col: "pi"})

print(f"\n  Rows: {len(df_pi):,}")
print(f"  Occupations: {df_pi['occ_id'].nunique():,}")
print(f"  Tasks: {df_pi['task_id'].nunique():,}")
print("\nTask-time distribution loaded ✓")

Loading task-time distribution …
  Detected 'occ' → 'O*NET-SOC Code'
  Detected 'task' → 'Task ID'
  Detected 'pi' → 'pi'

  Rows: 17,890
  Occupations: 894
  Tasks: 17,890

Task-time distribution loaded ✓


In [21]:
# ── 2.3 Task exposure labels ──────────────────────────────────────────────────
print("Loading task exposure labels …", flush=True)
df_exp = pd.read_csv(EXPOSURE_PATH, sep="\t")

# Auto-detect columns
EXP_OCC_CANDIDATES   = ["O*NET-SOC Code", "onet_code", "onet", "soc_code"]
EXP_TASK_CANDIDATES  = ["Task ID", "task_id", "TaskID"]
EXP_LABEL_CANDIDATES = ["new_labels", "exposure_label", "label", "gpt4_exposure", "human_exposure_agg"]

exp_occ_col   = next((c for c in EXP_OCC_CANDIDATES   if c in df_exp.columns), None)
exp_task_col  = next((c for c in EXP_TASK_CANDIDATES  if c in df_exp.columns), None)
exp_label_col = next((c for c in EXP_LABEL_CANDIDATES if c in df_exp.columns), None)

for label, val, cands in [
    ("occ",   exp_occ_col,   EXP_OCC_CANDIDATES),
    ("task",  exp_task_col,  EXP_TASK_CANDIDATES),
    ("label", exp_label_col, EXP_LABEL_CANDIDATES),
]:
    if val is None:
        raise KeyError(f"Exposure file: cannot find column for '{label}'. "
                       f"Expected one of {cands}. Got: {df_exp.columns.tolist()}")
    print(f"  Detected '{label}' → '{val}'")

# Keep only needed columns and rename
df_exp = df_exp[[exp_occ_col, exp_task_col, exp_label_col]].copy()
df_exp.columns = ["occ_id", "task_id", "exposure_label"]

print(f"\n  Rows: {len(df_exp):,}")
print(f"\n  Exposure label distribution:")
print(df_exp["exposure_label"].value_counts().to_string())
print("\nExposure labels loaded ✓")

Loading task exposure labels …
  Detected 'occ' → 'O*NET-SOC Code'
  Detected 'task' → 'Task ID'
  Detected 'label' → 'new_labels'

  Rows: 19,265

  Exposure label distribution:
exposure_label
E0    10206
E1     4677
E2     3327
E3      519

Exposure labels loaded ✓


In [22]:
# ── 2.4 Observed task penetration (Anthropic) ────────────────────────────────
print("Loading observed task penetration data …", flush=True)
df_pen = pd.read_csv(PENETRATION_PATH)
df_pen["penetration"] = pd.to_numeric(df_pen["penetration"], errors="coerce").fillna(0)
df_pen["task_clean"] = df_pen["task"].astype(str).str.strip().str.lower()

pen_lookup = (
    df_pen.groupby("task_clean")["penetration"]
    .max()
    .reset_index()
)

# Map task_id → task text via the full labelset (each Task ID has one unique text)
df_task_text = pd.read_csv(EXPOSURE_PATH, sep="\t", usecols=["Task ID", "Task"])
df_task_text = df_task_text.drop_duplicates(subset=["Task ID"])
df_task_text.columns = ["task_id", "task_text"]
df_task_text["task_clean"] = df_task_text["task_text"].astype(str).str.strip().str.lower()

# Build task_id → penetration
df_task_pen = df_task_text.merge(pen_lookup, on="task_clean", how="left")
df_task_pen["penetration"] = df_task_pen["penetration"].fillna(0)
pen_by_task_id = df_task_pen.set_index("task_id")["penetration"]

n_with_pen = (pen_by_task_id > 0).sum()
print(f"  Unique tasks in penetration file : {len(df_pen):,}")
print(f"  Tasks mapped via Task ID         : {len(pen_by_task_id):,}")
print(f"  Tasks with observed usage (>0)   : {n_with_pen:,} ({n_with_pen/len(pen_by_task_id):.1%})")
print("\nObserved penetration data loaded ✓")

Loading observed task penetration data …
  Unique tasks in penetration file : 18,118
  Tasks mapped via Task ID         : 19,265
  Tasks with observed usage (>0)   : 1,735 (9.0%)

Observed penetration data loaded ✓


## 3 · Aggregate Workforce → Occupation-Level Worker Counts and Wages

In [23]:
# ── 3.1  Drop rows missing the occupation identifier ─────────────────────────
before = len(df_raw)
df_work = df_raw.dropna(subset=[OCC_COL]).copy()
after  = len(df_work)
print(f"Dropped {before - after:,} rows with missing occupation code "
      f"({(before - after) / before:.1%} of total)")

# ── 3.2  Impute missing wages from occupation median (weighted) ───────────────
# First compute weighted median salary per onet_code for those that do have one
df_work[WEIGHT_COL] = df_work[WEIGHT_COL].fillna(1.0)     # treat null weight as 1

occ_median_wage = (
    df_work.dropna(subset=[WAGE_COL])
    .groupby(OCC_COL)
    .apply(lambda g: np.average(g[WAGE_COL], weights=g[WEIGHT_COL]))
    .rename("median_occ_wage")
    .reset_index()
)
occ_median_wage.columns = [OCC_COL, "median_occ_wage"]

df_work = df_work.merge(occ_median_wage, on=OCC_COL, how="left")
df_work[WAGE_COL] = df_work[WAGE_COL].fillna(df_work["median_occ_wage"])

n_still_missing = df_work[WAGE_COL].isna().sum()
print(f"After occupation-level imputation: {n_still_missing:,} rows still missing wage "
      f"(these will be excluded from wage mass).")

# ── 3.3  Occupation-level aggregation ────────────────────────────────────────
# n_workers = sum of weights (corrected headcount)
# avg_wage  = weighted mean of salary
def wtd_mean(g):
    mask = g[WAGE_COL].notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(g.loc[mask, WAGE_COL], weights=g.loc[mask, WEIGHT_COL])

occ_agg = (
    df_work.groupby(OCC_COL)
    .apply(lambda g: pd.Series({
        "n_workers": g[WEIGHT_COL].sum(),
        "avg_wage" : wtd_mean(g),
    }))
    .reset_index()
)
occ_agg.columns = ["occ_id", "n_workers", "avg_wage"]

# Drop occupations with no usable wage data
before_occ = len(occ_agg)
occ_agg = occ_agg.dropna(subset=["avg_wage"])
after_occ = len(occ_agg)
print(f"\nOccupations after aggregation : {before_occ:,}  (dropped {before_occ - after_occ:,} with no wage data)")

# ── 3.4  Sanity-check wage mass ───────────────────────────────────────────────
total_wage_mass = (occ_agg["n_workers"] * occ_agg["avg_wage"]).sum()
print(f"\nTotal workforce wage mass  :  ${total_wage_mass:,.0f}")
print(f"Occupations covered        :  {len(occ_agg):,}")
print(f"Weighted workers covered   :  {occ_agg['n_workers'].sum():,.0f}")
print("\nOccupation aggregation ✓")

Dropped 10,767 rows with missing occupation code (0.0% of total)
After occupation-level imputation: 0 rows still missing wage (these will be excluded from wage mass).

Occupations after aggregation : 1,010  (dropped 0 with no wage data)

Total workforce wage mass  :  $1,903,887,184,468
Occupations covered        :  1,010
Weighted workers covered   :  26,852,918

Occupation aggregation ✓


## 4 · Build Occupation × Task Table

In [24]:
# ── 4.1  Join occupation aggregates → task-time table ────────────────────────
# Inner join: only occupations that appear in both the workforce AND the π table
df_occ_task = df_pi.merge(occ_agg, on="occ_id", how="inner")

n_occ_matched   = df_occ_task["occ_id"].nunique()
n_occ_workforce = occ_agg["occ_id"].nunique()
n_occ_pi        = df_pi["occ_id"].nunique()

print(f"Occupations in workforce data  : {n_occ_workforce:,}")
print(f"Occupations in π table         : {n_occ_pi:,}")
print(f"Occupations matched (inner)    : {n_occ_matched:,}")
print(f"Task rows after join            : {len(df_occ_task):,}")

# Wage mass retained after occupation matching
wage_mass_matched = (df_occ_task.groupby("occ_id")
                     .first()[["n_workers", "avg_wage"]]
                     .eval("wage_mass = n_workers * avg_wage")
                     ["wage_mass"].sum())
print(f"\nWage mass in matched occupations  : ${wage_mass_matched:,.0f}")
print(f"Share of total wage mass          : {wage_mass_matched / total_wage_mass:.1%}")

# ── 4.2  Attach exposure labels ───────────────────────────────────────────────
df_occ_task = df_occ_task.merge(df_exp, on=["occ_id", "task_id"], how="left")

n_unlabelled = df_occ_task["exposure_label"].isna().sum()
print(f"\nTask-rows without exposure label  : {n_unlabelled:,} "
      f"({n_unlabelled / len(df_occ_task):.1%})")

# Drop unlabelled tasks (cannot assign exposure bucket)
df_occ_task = df_occ_task.dropna(subset=["exposure_label"])
print(f"Task-rows after dropping unlabelled: {len(df_occ_task):,}")

print("\nOccupation × task table built ✓")

# ── 4.3  Attach observed penetration flag ─────────────────────────────
df_occ_task["penetration"] = df_occ_task["task_id"].map(pen_by_task_id).fillna(0)
df_occ_task["observed"] = (df_occ_task["penetration"] > 0).astype(int)

n_obs = df_occ_task["observed"].sum()
print(f"\nTask-rows with observed AI usage    : {n_obs:,} ({n_obs/len(df_occ_task):.1%})")


Occupations in workforce data  : 1,010
Occupations in π table         : 894
Occupations matched (inner)    : 888
Task rows after join            : 17,741

Wage mass in matched occupations  : $1,633,868,983,664
Share of total wage mass          : 85.8%

Task-rows without exposure label  : 65 (0.4%)
Task-rows after dropping unlabelled: 17,676

Occupation × task table built ✓

Task-rows with observed AI usage    : 1,614 (9.1%)


## 5 · Diagnostics

In [25]:
# ── 5.1  π sums per occupation (should all equal 1) ──────────────────────────
pi_sums = df_occ_task.groupby("occ_id")["pi"].sum()

print("=== Task-share (π) sum per occupation ===")
print(f"  Min  : {pi_sums.min():.4f}")
print(f"  Max  : {pi_sums.max():.4f}")
print(f"  Mean : {pi_sums.mean():.4f}")
print(f"  Occupations where |π_sum − 1| > 0.01 : "
      f"{(pi_sums - 1).abs().gt(0.01).sum()}")
print()

# NOTE: Due to unlabelled tasks being dropped, some occupations' π sums may be
#       slightly below 1.  We re-normalise within the labelled task set so that
#       the total task value reconstructs the occupation wage mass.
df_occ_task["pi_norm"] = df_occ_task.groupby("occ_id")["pi"].transform(
    lambda x: x / x.sum()
)
pi_sums_renorm = df_occ_task.groupby("occ_id")["pi_norm"].sum()
print(f"  After re-normalisation — max |π_sum − 1| : "
      f"{(pi_sums_renorm - 1).abs().max():.6f}")

# ── 5.2  Task counts per exposure class ──────────────────────────────────────
print("\n=== Task rows per exposure class ===")
print(df_occ_task["exposure_label"].value_counts().to_string())

print("\nDiagnostics ✓")

=== Task-share (π) sum per occupation ===
  Min  : 0.3246
  Max  : 1.0000
  Mean : 0.9965
  Occupations where |π_sum − 1| > 0.01 : 38

  After re-normalisation — max |π_sum − 1| : 0.000000

=== Task rows per exposure class ===
exposure_label
E0    9718
E1    4325
E2    3133
E3     500

Diagnostics ✓


## 6 · Compute Task Economic Value

In [26]:
# ── Task value = workers × avg_wage × π_norm ─────────────────────────────────
# (pi_norm is the re-normalised share that accounts for tasks with missing labels)
df_occ_task["task_value"] = (
    df_occ_task["n_workers"]
    * df_occ_task["avg_wage"]
    * df_occ_task["pi_norm"]
)

# ── Diagnostic: task values should sum to the occupation wage mass ────────────
reconstructed_wage_mass = df_occ_task["task_value"].sum()

print("=== Wage-mass reconstruction check ===")
print(f"  True occupation wage mass (matched occs)  : ${wage_mass_matched:,.0f}")
print(f"  Sum of task values                         : ${reconstructed_wage_mass:,.0f}")
diff = abs(reconstructed_wage_mass - wage_mass_matched)
print(f"  Absolute difference                        : ${diff:,.0f}  "
      f"({diff / wage_mass_matched:.4%})")
if diff / wage_mass_matched < 1e-6:
    print("  ✓  Task values perfectly reconstruct wage mass")
else:
    print("  ⚠  Small difference expected due to re-normalisation over labelled tasks")

print(f"\nTotal task-value rows  : {len(df_occ_task):,}")
print("\nTask economic values computed ✓")

=== Wage-mass reconstruction check ===
  True occupation wage mass (matched occs)  : $1,633,868,983,664
  Sum of task values                         : $1,633,868,983,664
  Absolute difference                        : $0  (0.0000%)
  ✓  Task values perfectly reconstruct wage mass

Total task-value rows  : 17,676

Task economic values computed ✓


## 7 · Assign Exposure Buckets and Compute Baseline Totals

In [27]:
# ── Map E2 and E3 → E2/E3 bucket ──────────────────────────────────────────────
BUCKET_MAP = {"E0": "E0", "E1": "E1", "E2": "E2/E3", "E3": "E2/E3"}

df_occ_task["bucket"] = df_occ_task["exposure_label"].map(BUCKET_MAP)

unmapped = df_occ_task["bucket"].isna().sum()
if unmapped > 0:
    unknown_labels = df_occ_task.loc[df_occ_task["bucket"].isna(), "exposure_label"].unique()
    print(f"⚠  {unmapped:,} rows have unrecognised exposure labels: {unknown_labels}")
    df_occ_task = df_occ_task.dropna(subset=["bucket"])

# ── Baseline totals per bucket ────────────────────────────────────────────────
baseline = (
    df_occ_task.groupby("bucket")["task_value"]
    .sum()
    .reindex(["E0", "E1", "E2/E3"])
    .fillna(0)
    .rename("baseline_value")
)

total_value = baseline.sum()

print("=== Baseline wage value by exposure class ===")
print(f"{'Class':<10} {'Value ($)':>18} {'Share':>8}")
print("-" * 40)
for bucket, val in baseline.items():
    print(f"{bucket:<10} ${val:>17,.0f} {val / total_value:>7.1%}")
print("-" * 40)
print(f"{'TOTAL':<10} ${total_value:>17,.0f} {'100.0%':>8}")

print("\nExposure bucket totals ✓")

=== Baseline wage value by exposure class ===
Class               Value ($)    Share
----------------------------------------
E0         $  368,480,425,899   22.6%
E1         $  664,799,624,921   40.7%
E2/E3      $  600,588,932,844   36.8%
----------------------------------------
TOTAL      $1,633,868,983,664   100.0%

Exposure bucket totals ✓


## 8 · Automation Scenarios (40 %, 50 %, 60 %)

The Gemini-rubric pipeline is unchanged; only the scenario rates are updated. The **50 %** scenario
is used as the headline summary for the E1 + E2/E3 wage mass.

In [28]:
# ── Automation rates ──────────────────────────────────────────────────────────
SCENARIOS = {"40%": 0.40, "50%": 0.50, "60%": 0.60}
HEADLINE_RATE_LABEL = "50%"

# E0 is never automated
val_E0   = baseline["E0"]
val_E1   = baseline["E1"]
val_E2_3 = baseline["E2/E3"]

rows = []
for label, rate in SCENARIOS.items():
    auto_E1   = val_E1   * rate
    auto_E2_3 = val_E2_3 * rate
    rows.append({
        "Scenario"          : f"Scenario {label}",
        "Rate"              : rate,
        "Auto E1 ($)"       : auto_E1,
        "Auto E2/E3 ($)"    : auto_E2_3,
        "Total Auto ($)"    : auto_E1 + auto_E2_3,
        "Auto Share (%)"    : (auto_E1 + auto_E2_3) / total_value * 100,
    })

df_scenarios = pd.DataFrame(rows)

print("=== Automation scenario results ===\n")
print(df_scenarios.to_string(index=False, float_format="${:,.0f}".format))

print("\nAutomation scenarios ✓")

=== Automation scenario results ===

    Scenario  Rate      Auto E1 ($)   Auto E2/E3 ($)   Total Auto ($)  Auto Share (%)
Scenario 40%    $0 $265,919,849,968 $240,235,573,138 $506,155,423,106             $31
Scenario 50%    $0 $332,399,812,461 $300,294,466,422 $632,694,278,883             $39
Scenario 60%    $1 $398,879,774,953 $360,353,359,707 $759,233,134,659             $46

Automation scenarios ✓


## 9 · Summary Table

In [29]:
# ── Build composite summary table ────────────────────────────────────────────
# Columns: Class | Baseline Value | {40,50,60}% Automation

buckets = ["E0", "E1", "E2/E3"]
scenario_cols = [f"{label} Automation ($)" for label in SCENARIOS.keys()]

summary_rows = []
scenarios_indexed = df_scenarios.set_index("Scenario")

for b in buckets:
    bval = baseline[b]
    row = {"Class": b, "Baseline Value ($)": bval}
    if b == "E0":
        for col in scenario_cols:
            row[col] = 0
    else:
        src_col = "Auto E1 ($)" if b == "E1" else "Auto E2/E3 ($)"
        for label in SCENARIOS.keys():
            row[f"{label} Automation ($)"] = scenarios_indexed.loc[f"Scenario {label}", src_col]
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# ── Print the summary table ───────────────────────────────────────────────────
def fmt_usd(x):
    return f"${x:,.0f}"

print("=" * 90)
print("  TASK-VALUE EXPOSURE SUMMARY  (S&P 500 workforce)")
print("=" * 90)
print(df_summary.to_string(index=False, float_format="${:,.0f}".format))
print("=" * 90)

print(f"\n  Total workforce wage mass represented  : ${total_wage_mass:,.0f}")
print(f"  Wage mass in matched occupations       : ${wage_mass_matched:,.0f}   "
      f"({wage_mass_matched / total_wage_mass:.1%} of total)")
print()

print("  Share of total task value by exposure class:")
for b in buckets:
    print(f"    {b:<6} : {baseline[b] / total_value:>6.1%}")
print()

print("  Share of potentially automated value (E1 + E2/E3):")
for _, srow in df_scenarios.iterrows():
    mark = "  ← headline" if srow["Scenario"] == f"Scenario {HEADLINE_RATE_LABEL}" else ""
    print(f"    {srow['Scenario']:<16}: {srow['Auto Share (%)']:>5.1f}% of total task value{mark}")

# ── Headline summary: E1 + E2/E3 at 50% ──────────────────────────────────────
headline_rate = SCENARIOS[HEADLINE_RATE_LABEL]
rubric_exposed_value = baseline["E1"] + baseline["E2/E3"]
headline_auto = rubric_exposed_value * headline_rate

print()
print(f"  Headline (E1 + E2/E3 @ {HEADLINE_RATE_LABEL}):")
print(f"    Exposed wage mass   : ${rubric_exposed_value:,.0f}")
print(f"    Automated wage mass : ${headline_auto:,.0f}   "
      f"({headline_auto / total_value:.1%} of total task value)")

print()
print("Summary ✓")

  TASK-VALUE EXPOSURE SUMMARY  (S&P 500 workforce)
Class  Baseline Value ($)  40% Automation ($)  50% Automation ($)  60% Automation ($)
   E0    $368,480,425,899                  $0                  $0                  $0
   E1    $664,799,624,921    $265,919,849,968    $332,399,812,461    $398,879,774,953
E2/E3    $600,588,932,844    $240,235,573,138    $300,294,466,422    $360,353,359,707

  Total workforce wage mass represented  : $1,903,887,184,468
  Wage mass in matched occupations       : $1,633,868,983,664   (85.8% of total)

  Share of total task value by exposure class:
    E0     :  22.6%
    E1     :  40.7%
    E2/E3  :  36.8%

  Share of potentially automated value (E1 + E2/E3):
    Scenario 40%    :  31.0% of total task value
    Scenario 50%    :  38.7% of total task value  ← headline
    Scenario 60%    :  46.5% of total task value

  Headline (E1 + E2/E3 @ 50%):
    Exposed wage mass   : $1,265,388,557,765
    Automated wage mass : $632,694,278,883   (38.7% of total ta

## 10 · Top Occupations Contributing to E1 and E2/E3 Exposure

In [30]:
# Build occupation-level task-value aggregates with titles where available
occ_bucket_value = (
    df_occ_task.groupby(["occ_id", "bucket"])["task_value"]
    .sum()
    .reset_index()
)

# Attach occupation title from the π table (if it has a Title column)
title_map = None
if "Title" in df_pi.columns or "title" in df_pi.columns:
    title_col = "Title" if "Title" in df_pi.columns else "title"
    title_map = df_pi[["occ_id", title_col]].drop_duplicates().rename(columns={title_col: "occ_title"})
    occ_bucket_value = occ_bucket_value.merge(title_map, on="occ_id", how="left")
else:
    occ_bucket_value["occ_title"] = occ_bucket_value["occ_id"]

TOP_N = 15

for bucket in ["E1", "E2/E3"]:
    sub = (
        occ_bucket_value[occ_bucket_value["bucket"] == bucket]
        .sort_values("task_value", ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    sub.index += 1
    sub["Share of bucket (%)"] = sub["task_value"] / baseline[bucket] * 100
    print(f"\n{'='*65}")
    print(f"  Top {TOP_N} occupations by task value in  {bucket}")
    print(f"{'='*65}")
    cols = ["occ_id", "occ_title", "task_value", "Share of bucket (%)"]
    print(sub[cols].to_string(float_format="${:,.0f}".format))


  Top 15 occupations by task value in  E1
        occ_id                                        occ_title       task_value  Share of bucket (%)
1   15-1252.00                              Software Developers $122,129,887,640                  $18
2   11-2021.00                               Marketing Managers  $50,435,721,764                   $8
3   15-1299.08            Computer Systems Engineers/Architects  $42,386,254,709                   $6
4   15-1299.09          Information Technology Project Managers  $26,510,017,212                   $4
5   11-2022.00                                   Sales Managers  $20,304,066,276                   $3
6   15-2051.01                   Business Intelligence Analysts  $19,534,183,697                   $3
7   11-1011.00                                 Chief Executives  $16,921,333,676                   $3
8   11-3031.03                         Investment Fund Managers  $14,751,548,346                   $2
9   15-1243.00                         

## 11 · Observed AI Usage — Penetration as Efficiency Proxy

The Gemini-rubric analysis (Sections 7–9) assumes a flat automation rate within each exposure
bucket. This section replaces that assumption with a **data-driven efficiency measure** derived
directly from the Anthropic task-penetration data.

**Method.** For each occupation × task row we treat the penetration value as the fraction of that
task's dollar value that is already captured (or captureable) by AI. The per-row dollar saving is:

```
savings_{occ,task} = task_value_{occ,task} × penetration_task
                   = (n_workers × avg_wage × π_norm) × penetration_task
```

Summing over the full SP500 occupation × task grid yields the total observed efficiency saving.
Penetration therefore plays the same structural role that the flat automation rate plays on the
Gemini side, but it varies task-by-task using the observed weighted usage across the six Anthropic
snapshots. The total savings can be read equivalently as the **wage-weighted mean penetration**
times the total task wage mass.

In [31]:
# ── Penetration-weighted efficiency savings ──────────────────────────────────
# task_value = n_workers (weight) × avg_wage × pi_norm
# penetration ∈ [0, 1] acts as a per-task efficiency coefficient.
# Per-row savings = task_value × penetration.

df_occ_task["obs_savings"] = df_occ_task["task_value"] * df_occ_task["penetration"]

obs_savings_total = df_occ_task["obs_savings"].sum()
obs_share_total   = obs_savings_total / total_value

# Equivalent wage-weighted mean penetration across the SP500 task set.
wage_weighted_pen = obs_savings_total / total_value

print("=" * 80)
print("  OBSERVED AI USAGE — Penetration-weighted efficiency savings")
print("=" * 80)
print(f"  Total SP500 task wage mass                         : ${total_value:,.0f}")
print(f"  Penetration-weighted $ savings (Σ task_value × pen): ${obs_savings_total:,.0f}")
print(f"  Share of total task wage mass                      : {obs_share_total:.2%}")
print(f"  Wage-weighted mean penetration (economy-wide)      : {wage_weighted_pen:.2%}")
print("=" * 80)

# ── Breakdown by Gemini exposure bucket (informational) ───────────────────────
savings_by_bucket = (
    df_occ_task.groupby("bucket")["obs_savings"]
    .sum()
    .reindex(["E0", "E1", "E2/E3"])
    .fillna(0)
)

print("\n  Savings breakdown by rubric bucket:")
print(f"  {'Bucket':<8} {'Savings ($)':>18} {'% of savings':>14} {'% of bucket value':>20}")
print("  " + "-" * 62)
for b, v in savings_by_bucket.items():
    pct_of_savings = v / obs_savings_total if obs_savings_total else 0.0
    pct_of_bucket  = v / baseline[b]       if baseline[b]      else 0.0
    print(f"  {b:<8} ${v:>17,.0f} {pct_of_savings:>13.1%} {pct_of_bucket:>19.2%}")

# ── Exposed wage mass (tasks with penetration > 0) ───────────────────────────
# Analogous to the Gemini side's E1 + E2/E3 bucket: tasks with penetration == 0
# are not exposed under the observed method, so they must not be counted in the
# "exposed" denominator.
n_rows_with_pen    = int((df_occ_task["penetration"] > 0).sum())
obs_exposed_value  = df_occ_task.loc[df_occ_task["penetration"] > 0, "task_value"].sum()
obs_unexposed_val  = total_value - obs_exposed_value
implied_pen_on_exp = obs_savings_total / obs_exposed_value if obs_exposed_value > 0 else 0.0

print(f"\n  Exposed wage mass  (tasks with penetration > 0) : ${obs_exposed_value:,.0f}   "
      f"({obs_exposed_value/total_value:.1%} of total)")
print(f"  Unexposed wage mass (penetration == 0)          : ${obs_unexposed_val:,.0f}   "
      f"({obs_unexposed_val/total_value:.1%} of total)")
print(f"  Task-rows with penetration > 0                  : {n_rows_with_pen:,} "
      f"({n_rows_with_pen/len(df_occ_task):.1%} of rows)")
print(f"  Wage-weighted mean penetration on exposed set   : {implied_pen_on_exp:.2%}")
print(f"  Savings / exposed wage mass                     : "
      f"{obs_savings_total/obs_exposed_value:.2%}" if obs_exposed_value > 0
      else "  Savings / exposed wage mass                     : n/a")

print("\nObserved penetration-weighted savings ✓")

  OBSERVED AI USAGE — Penetration-weighted efficiency savings
  Total SP500 task wage mass                         : $1,633,868,983,664
  Penetration-weighted $ savings (Σ task_value × pen): $327,544,267,237
  Share of total task wage mass                      : 20.05%
  Wage-weighted mean penetration (economy-wide)      : 20.05%

  Savings breakdown by rubric bucket:
  Bucket          Savings ($)   % of savings    % of bucket value
  --------------------------------------------------------------
  E0       $   19,870,413,371          6.1%               5.39%
  E1       $  175,628,004,330         53.6%              26.42%
  E2/E3    $  132,045,849,536         40.3%              21.99%

  Exposed wage mass  (tasks with penetration > 0) : $369,957,468,958   (22.6% of total)
  Unexposed wage mass (penetration == 0)          : $1,263,911,514,706   (77.4% of total)
  Task-rows with penetration > 0                  : 1,614 (9.1% of rows)
  Wage-weighted mean penetration on exposed set   : 88

## 12 · Combined Summary — Gemini Rubric vs. Observed Penetration

Side-by-side view of the two approaches:

- **Gemini rubric**: flat automation rates of 40 %, 50 %, 60 % applied to E1 + E2/E3 wage mass.
  The **50 %** scenario is the headline figure.
- **Observed**: the penetration-weighted dollar saving Σ(task_value × penetration) across all
  SP500 tasks. There is no flat automation rate; each task contributes proportionally to its
  observed penetration.

In [32]:
# ── Combined summary: Gemini rubric vs. Observed penetration ─────────────────
rubric_exposed_value = baseline["E1"] + baseline["E2/E3"]
headline_rate = SCENARIOS[HEADLINE_RATE_LABEL]
headline_auto = rubric_exposed_value * headline_rate

print("=" * 110)
print("  COMBINED SUMMARY — Gemini Rubric vs. Observed Penetration")
print("=" * 110)

# ── Gemini rubric scenarios ──
print(f"\n  {'--- Gemini rubric (flat rate on E1 + E2/E3) ---':^72}")
print(f"  {'Scenario':>12}  {'Exposed ($)':>18}  {'Automated ($)':>18}  {'Share of total':>14}")
print("  " + "-" * 72)
for label, rate in SCENARIOS.items():
    rubric_auto = rubric_exposed_value * rate
    marker = "  ← headline" if label == HEADLINE_RATE_LABEL else ""
    print(f"  {label:>12}  ${rubric_exposed_value:>17,.0f}  ${rubric_auto:>17,.0f}  "
          f"{rubric_auto / total_value:>13.1%}{marker}")

# ── Observed (penetration-weighted) ──
# "Exposed" here = wage mass of tasks with penetration > 0, analogous to the
# Gemini side's E1 + E2/E3 mass. Tasks with penetration == 0 are NOT exposed.
print(f"\n  {'--- Observed (penetration as efficiency proxy) ---':^72}")
print(f"  {'Method':>12}  {'Exposed ($)':>18}  {'Savings ($)':>18}  {'Share of total':>14}")
print("  " + "-" * 72)
print(f"  {'Σ val×pen':>12}  ${obs_exposed_value:>17,.0f}  ${obs_savings_total:>17,.0f}  "
      f"{obs_savings_total / total_value:>13.1%}")
print(f"  (effective automation on exposed mass: "
      f"{obs_savings_total / obs_exposed_value:.1%})"
      if obs_exposed_value > 0 else
      "  (effective automation on exposed mass: n/a)")

# ── Headline comparison ──
print("\n" + "=" * 110)
print(f"  Total SP500 task wage mass                      : ${total_value:,.0f}")
print(f"  Gemini exposed mass (E1 + E2/E3)                : "
      f"${rubric_exposed_value:,.0f}   ({rubric_exposed_value / total_value:.1%} of total)")
print(f"  Observed exposed mass (penetration > 0)         : "
      f"${obs_exposed_value:,.0f}   ({obs_exposed_value / total_value:.1%} of total)")
print(f"  Gemini headline (E1 + E2/E3 @ {HEADLINE_RATE_LABEL})  savings : "
      f"${headline_auto:,.0f}   ({headline_auto / total_value:.2%} of total)")
print(f"  Observed (penetration-weighted)          savings : "
      f"${obs_savings_total:,.0f}   ({obs_savings_total / total_value:.2%} of total)")

delta = headline_auto - obs_savings_total
ratio = headline_auto / obs_savings_total if obs_savings_total else float("nan")
print(f"\n  Gemini 50%  −  Observed                          : "
      f"${delta:,.0f}   (Gemini / Observed = {ratio:,.2f}×)")

print("\nCombined summary ✓")

  COMBINED SUMMARY — Gemini Rubric vs. Observed Penetration

              --- Gemini rubric (flat rate on E1 + E2/E3) ---             
      Scenario         Exposed ($)       Automated ($)  Share of total
  ------------------------------------------------------------------------
           40%  $1,265,388,557,765  $  506,155,423,106          31.0%
           50%  $1,265,388,557,765  $  632,694,278,883          38.7%  ← headline
           60%  $1,265,388,557,765  $  759,233,134,659          46.5%

             --- Observed (penetration as efficiency proxy) ---           
        Method         Exposed ($)         Savings ($)  Share of total
  ------------------------------------------------------------------------
     Σ val×pen  $  369,957,468,958  $  327,544,267,237          20.0%
  (effective automation on exposed mass: 88.5%)

  Total SP500 task wage mass                      : $1,633,868,983,664
  Gemini exposed mass (E1 + E2/E3)                : $1,265,388,557,765   (77.4% of t